<a href="https://colab.research.google.com/github/MiguelUTEC26/etl-data-pipeline-2534562019/blob/main/B_proveedores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install sqlalchemy psycopg2-binary
from sqlalchemy import create_engine
import pandas as pd
url = "https://raw.githubusercontent.com/MiguelUTEC26/etl-data-pipeline-2534562019/refs/heads/main/raw/B_proveedores.csv"
df = pd.read_csv(url)

df.head()

,id_proveedor,proveedor,pais
0,P300,SurtiMax 0,Guatemala
1,P301,Insumos Globales 1,Costa Rica
2,P302,Distribuidora Atlas 2,El Salvador
3,P303,TecnoSupply 3,Guatemala
4,P304,Insumos Globales 4,Guatemala


In [3]:
df.shape
df.columns
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id_proveedor  38 non-null     object
 1   proveedor     38 non-null     object
 2   pais          38 non-null     object
dtypes: object(3)
memory usage: 1.0+ KB


,0
id_proveedor,0
proveedor,0
pais,0


In [4]:
B_proveedores = df.copy()

B_proveedores.columns = B_proveedores.columns.str.strip().str.lower()

for col in B_proveedores.select_dtypes(include='object').columns:
    B_proveedores[col] = B_proveedores[col].astype(str).str.strip()

    B_proveedores = B_proveedores.replace(r'^\s*$', pd.NA, regex=True)

    B_proveedores = B_proveedores.drop_duplicates()

In [5]:
validos = B_proveedores[
    B_proveedores['id_proveedor'].notna() &
    B_proveedores['proveedor'].notna() &
    B_proveedores['pais'].notna()
].copy()

In [6]:
rechazados = B_proveedores[
    B_proveedores['id_proveedor'].isna() |
    B_proveedores['proveedor'].isna()|
    B_proveedores['pais'].isna()
].copy()

In [7]:
def motivo(row):

    motivos = []

    if pd.isna(row['id_proveedor']):
        motivos.append("id_proveedor_vacio")

    if pd.isna(row['proveedor']):
        motivos.append("proveedor_vacio")

    if pd.isna(row['pais']):
        motivos.append("pais_vacio")

    return ",".join(motivos)

rechazados["motivo_rechazo"] = rechazados.apply(motivo, axis=1)

In [8]:
rechazados.to_csv("B_proveedores_rejects.csv", index=False)
validos.to_csv("B_proveedores_curated.csv", index=False)